In [ ]:
import json
import random
import requests
import pandas as pd
import gradio as gr
from pydantic import BaseModel, Field

In [ ]:
# --- CONFIGURATION ---
OLLAMA_URL = "http://localhost:11434/api/generate"
DEFAULT_MODEL = "llama3.2"  # Ensure you pulled this model in Ollama (`ollama pull llama3.2`)

GRADES = [f"Grade {i}" for i in range(1, 11)]
SUBJECTS = ["Mathematics", "Science", "English Literature", "History", "Physics", "Chemistry"]
COUNTRIES = ["India", "United States", "United Kingdom", "Nigeria", "Brazil", "Japan"]

In [ ]:
# --- PYDANTIC SCHEMA FOR STRUCTURED OUTPUT ---
class StudentRecord(BaseModel):
    grade: str
    country: str
    subject: str
    failure_cause: str
    student_persona: str
    remedial_strategy: str

In [ ]:
# --- DATASET GENERATION FUNCTION ---
def generate_single_record(model_name: str) -> dict:
    selected_grade = random.choice(GRADES)
    selected_country = random.choice(COUNTRIES)
    selected_subject = random.choice(SUBJECTS)

    prompt = f"""
    Generate a realistic, empathetic student record for a student who failed an exam.
    Grade: {selected_grade}
    Country: {selected_country}
    Subject: {selected_subject}

    Provide:
    1. Primary cause of failure (e.g., connection between student and teacher, interest of student, complexity of language).
    2. A brief 2-sentence profile of the student.
    3. What does curriculum needs to change to make it accessible for students.

    You realise that student do not lack anything, all we need is to make the knowledge accessible to them in a way for them to observe and understand, so they can apply in their lives.
    """

    payload = {
        "model": model_name,
        "prompt": prompt,
        "format": StudentRecord.model_json_schema(), # Forces JSON output
        "stream": False
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()
        result_text = response.json().get("response", "{}")
        return json.loads(result_text)
    except Exception as e:
        return {"error": str(e)}

In [ ]:
def build_dataset(model_name: str, num_records: int, progress=gr.Progress()):
    records = []
    for i in range(num_records):
        progress(i / num_records, desc=f"Generating record {i+1} of {num_records}...")
        data = generate_single_record(model_name)
        if "error" not in data:
            records.append(data)
    
    df = pd.DataFrame(records)
    csv_path = "generated_student_dataset.csv"
    df.to_csv(csv_path, index=False)
    
    return df, csv_path

In [ ]:
# --- GRADIO UI ---
with gr.Blocks(title="Synthetic Student Dataset Generator") as demo:
    gr.Markdown("# 🎓 Synthetic Student Failure & Remedial Dataset Generator")
    gr.Markdown("Uses local Ollama models to build an evaluation dataset for junior to 10th-grade students.")
    
    with gr.Row():
        model_input = gr.Textbox(value="llama3.2", label="Ollama Model Name")
        count_input = gr.Slider(minimum=1, maximum=50, value=5, step=1, label="Number of Records to Generate")
    
    generate_btn = gr.Button("Generate Dataset", variant="primary")
    
    output_table = gr.Dataframe(label="Generated Dataset Preview")
    file_output = gr.File(label="Download CSV")
    
    generate_btn.click(
        fn=build_dataset,
        inputs=[model_input, count_input],
        outputs=[output_table, file_output]
    )

    demo.launch()